# $D^0 \to K^+ K^- \pi^+ \pi^-$: the LHCb model from AmpGen

The amplitude model of LHCb, JHEP 02 (2019) 126, as defined in
[AmpGen](https://github.com/GooFit/AmpGen)'s `options/DtoKKpipi_v2.opt`, built as SymPy
expressions by `aman_examples.ampgen`. That module reads AmpGen options files and
follows AmpGen's conventions: daughter ordering, default orbital angular momenta,
covariant spin factors and AmpGen's lineshapes. Each of the 55 decay chains agrees with
AmpGen to about $10^{-5}$ of its rms (`tests/test_ampgen_d02kkpipi.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from IPython.display import Math
from tensorwaves.data import IntensityDistributionGenerator
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.estimator import UnbinnedNLL, create_cached_function
from tensorwaves.function.sympy import create_parametrized_function
from tensorwaves.optimizer import Minuit2

from aman_examples import d02kkpipi
from aman_examples.phasespace import ImportanceSampledFunction

## Model

The decay chains, named by their AmpGen descriptors:

In [ ]:
model = d02kkpipi.formulate_model()
for name in model.chains:
    print(name)

Each chain is a product of lineshapes (unevaluated AmpGen expressions) and a
covariant spin factor, expanded in the four-momentum components `p<i>_x`, ..., `p<i>_E`
of $(K^+, K^-, \pi^+, \pi^-)$. For example, the lineshapes of the
$D^0 \to \phi(1020)\,\rho(770)$ chain:

In [ ]:
chain = model.chains[
    "D0{rhoOmega00{rho(770)0[GounarisSakurai]{pi+,pi-}},phi(1020)0{K+,K-}}"
]
lineshapes = [
    f
    for f in sp.Mul.make_args(chain)
    if type(f).__name__ in {"AmpGenBreitWigner", "AmpGenGounarisSakurai", "FormFactor"}
]
for f in lineshapes:
    display(Math(sp.latex(f)))

## Toy fit

Toys are generated with TensorWaves from the intensity $|A|^2$ by accept-reject. In
uniform four-body phase space only about 0.5% of the events would be accepted: the
intensity is dominated by narrow peaks, the $\phi(1020)$ in $m(K^+K^-)$ and the
$\rho(770)$–$\omega(782)$ in $m(\pi^+\pi^-)$. The events are therefore drawn from
`d02kkpipi.peaked_phase_space_generator`, a `PairPhaseSpaceGenerator` that builds the
phase space as $D^0 \to (K^+K^-)(\pi^+\pi^-)$ and draws the pair masses partly from
Breit-Wigner densities at these resonances. Its events are distributed as phase space
times a known density ratio $r$, and accepting them with probability $\propto |A|^2/r$
(`ImportanceSampledFunction`) gives the same toys as from uniform phase space, about
ten times faster.

The 25 free $D^0$ couplings are then fitted. Only they change in the fit, so all chain
amplitudes are cached with `create_cached_function`. The normalisation integral is
also computed with events of the peaked generator, weighted by $1/r$ (the `weights` of
the phase-space sample of `UnbinnedNLL`): for this model its statistical precision is
that of a uniform sample three times larger, so the fit is three times faster.

In [ ]:
expression = model.intensity
intensity = create_parametrized_function(
    expression, model.parameter_defaults, backend="jax"
)
peaked = d02kkpipi.peaked_phase_space_generator(model)
rng = NumpyUniformRNG(seed=1)
data = IntensityDistributionGenerator(
    peaked, ImportanceSampledFunction(intensity, peaked), bunch_size=100_000
).generate(20_000, rng)
phsp = peaked.generate(350_000, rng)

In [ ]:
free = [c for c in model.free_couplings if c.name.startswith("C_{D0")]
cached_intensity, transformer = create_cached_function(
    expression, model.parameter_defaults, backend="jax", free_parameters=free
)
cached_data, cached_phsp = transformer(data), transformer(phsp)
cached_phsp["weights"] = 1 / peaked.density_ratio(phsp)

true = {c.name: model.parameter_defaults[c] for c in free}
shift = np.random.default_rng(2)
initial = {k: v * (1 + 0.2 * complex(*shift.normal(size=2))) for k, v in true.items()}
estimator = UnbinnedNLL(cached_intensity, cached_data, cached_phsp, backend="jax")
result = Minuit2().optimize(estimator, initial)
print(
    f"valid: {result.minimum_valid}, {result.function_calls} calls, {result.execution_time:.0f} s"
)

In [ ]:
pulls = []
for name, value in true.items():
    fitted, error = result.parameter_values[name], result.parameter_errors[name]
    pulls += [(fitted - value).real / error.real, (fitted - value).imag / error.imag]
pulls = np.array(pulls)
print(
    f"{len(pulls)} pulls: mean {pulls.mean():+.2f}, standard deviation {pulls.std():.2f}"
)
plt.hist(pulls, bins=np.linspace(-4, 4, 17))
plt.xlabel("pull")
plt.show()

## Projections

In [ ]:
intensity.update_parameters(result.parameter_values)
weights = intensity(phsp) / peaked.density_ratio(phsp)


def mass(sample, indices):
    p = sum(np.column_stack([sample[f"p{i}_{c}"] for c in "xyzE"]) for i in indices)
    return np.sqrt(p[:, 3] ** 2 - np.sum(p[:, :3] ** 2, axis=1))


projections = [
    ("$m(K^+K^-)$", (0, 1)),
    (r"$m(\pi^+\pi^-)$", (2, 3)),
    (r"$m(K^+\pi^-)$", (0, 3)),
    (r"$m(K^-\pi^+)$", (1, 2)),
]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), tight_layout=True)
for ax, (label, indices) in zip(axes, projections, strict=True):
    counts, edges = np.histogram(mass(data, indices), bins=80)
    fitted, _ = np.histogram(mass(phsp, indices), bins=edges, weights=weights)
    centers = (edges[1:] + edges[:-1]) / 2
    ax.errorbar(
        centers, counts, yerr=np.sqrt(counts), fmt=".", color="black", label="toys"
    )
    ax.step(centers, fitted * counts.sum() / fitted.sum(), where="mid", label="fit")
    ax.set_xlabel(f"{label} [GeV]")
axes[0].legend()
plt.show()